# Validasi 30 Data Testing: XGBoost-Tabular 3 Fitur vs XGBoost-ST+GNN Hybrid

Notebook ini dibuat untuk revisi validasi biologis berbasis data testing, bukan memakai seluruh `mutations.csv` sebagai data validasi.

Desain eksperimen:

- `X_test_with_metadata.csv` dipakai sebagai sumber 30 kasus validasi.
- Sampel validasi berisi 30 kasus testing: 15 resisten dan 15 sensitif, dipilih dengan `random_state=42`.
- XGBoost-Tabular dilatih ulang dari split training yang sama dengan 3 fitur murni: `Gene`, `drug`, dan `Mutation` yang di-encode ordinal.
- `mutations.csv` hanya dipakai untuk merekonstruksi split training dan melatih model tabular 3-fitur, bukan sebagai sumber validasi.
- XGBoost-ST+GNN Hybrid memakai model tersimpan `xgb_hybrid_gnn_sem.joblib` dan fitur testing 448 dimensi yang sudah ada di `X_test_with_metadata.csv`.


In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)


In [2]:
cwd = Path.cwd()
if (cwd / "X_test_with_metadata.csv").exists():
    DATA_DIR = cwd
elif (cwd / "Tugas Akhir" / "X_test_with_metadata.csv").exists():
    DATA_DIR = cwd / "Tugas Akhir"
else:
    raise FileNotFoundError("Tidak menemukan X_test_with_metadata.csv.")

MUTATIONS_CSV = DATA_DIR / "mutations.csv"
TEST_CSV = DATA_DIR / "X_test_with_metadata.csv"
HYBRID_MODEL_PATH = DATA_DIR / "xgb_hybrid_gnn_sem.joblib"

OUTPUT_DIR = DATA_DIR / "testing30_validation_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
N_PER_CLASS = 15

print("DATA_DIR:", DATA_DIR)
print("Output :", OUTPUT_DIR)


DATA_DIR: C:\Semester 7\Project Tugas Akhir\Tugas Akhir
Output : C:\Semester 7\Project Tugas Akhir\Tugas Akhir\testing30_validation_outputs


In [3]:
def map_confidence_to_binary(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    if "not assoc" in text:
        return 0
    if "assoc w r" in text:
        return 1
    return np.nan


def label_text(value):
    return "Resisten" if int(value) == 1 else "Sensitif"


def format_percent(value):
    return f"{value * 100:.1f}%".replace(".", ",")


In [4]:
# Load test set yang sudah disimpan dari split lama.
test_df = pd.read_csv(TEST_CSV)
feat_cols = sorted(
    [c for c in test_df.columns if c.startswith("feat_")],
    key=lambda x: int(x.split("_")[1]),
)

print("Test rows:", len(test_df))
print("Hybrid feature columns:", len(feat_cols))
display(test_df[["Gene", "Mutation", "drug", "confidence", "true_label"]].head())
display(test_df["true_label"].value_counts().sort_index().rename_axis("label").reset_index(name="n"))


Test rows: 2868
Hybrid feature columns: 448


,Gene,Mutation,drug,confidence,true_label
0,Rv0565c,c.351C>G,ethionamide,Not assoc w R - Interim,0
1,aftB,c.1782C>G,ethambutol,Not assoc w R - Interim,0
2,ubiA,c.690C>G,ethambutol,Not assoc w R - Interim,0
3,Rv2752c,c.1452C>T,rifampicin,Not assoc w R - Interim,0
4,dnaA,c.543C>A,isoniazid,Not assoc w R - Interim,0


,label,n
0,0,2573
1,1,295


In [5]:
# Rekonstruksi split training yang sama.
# Ini hanya untuk melatih ulang model tabular 3-fitur, bukan untuk validasi.
df = pd.read_csv(MUTATIONS_CSV)
df["label"] = df["confidence"].apply(map_confidence_to_binary)
df = df.dropna(subset=["label"]).reset_index(drop=True)
df["label"] = df["label"].astype(int)

indices = np.arange(len(df))
idx_train, idx_test = train_test_split(
    indices,
    test_size=0.2,
    stratify=df["label"],
    random_state=RANDOM_STATE,
)

train_df = df.loc[idx_train].reset_index(drop=True)
expected_test = df.loc[idx_test, ["Gene", "Mutation", "drug", "confidence", "label"]].reset_index(drop=True)
actual_test = test_df[["Gene", "Mutation", "drug", "confidence", "true_label"]].rename(columns={"true_label": "label"}).reset_index(drop=True)

metadata_match = (
    expected_test[["Gene", "Mutation", "drug"]].astype(str).equals(actual_test[["Gene", "Mutation", "drug"]].astype(str))
    and np.array_equal(expected_test["label"].to_numpy(), actual_test["label"].to_numpy())
)

print("Training rows:", len(train_df))
print("Testing rows :", len(actual_test))
print("Split metadata cocok dengan X_test_with_metadata.csv:", metadata_match)

if not metadata_match:
    raise ValueError("Split hasil rekonstruksi tidak cocok dengan X_test_with_metadata.csv.")


Training rows: 11470
Testing rows : 2868
Split metadata cocok dengan X_test_with_metadata.csv: True


In [6]:
# XGBoost-Tabular murni 3 fitur.
tabular_cols = ["Gene", "drug", "Mutation"]

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_train_tabular = encoder.fit_transform(train_df[tabular_cols].astype(str))
y_train = train_df["label"].to_numpy()

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_tabular = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_tabular.fit(X_train_tabular, y_train)

tabular_artifact_path = OUTPUT_DIR / "xgb_tabular_3fitur_retrained_from_train_split.joblib"
joblib.dump(
    {
        "model": xgb_tabular,
        "encoder": encoder,
        "feature_columns": tabular_cols,
        "random_state": RANDOM_STATE,
        "scale_pos_weight": float(scale_pos_weight),
    },
    tabular_artifact_path,
)

print("Tabular train feature shape:", X_train_tabular.shape)
print("Saved:", tabular_artifact_path)


Tabular train feature shape: (11470, 3)
Saved: C:\Semester 7\Project Tugas Akhir\Tugas Akhir\testing30_validation_outputs\xgb_tabular_3fitur_retrained_from_train_split.joblib


In [7]:
# Ambil 30 kasus testing: 15 resisten dan 15 sensitif.
resistant_cases = test_df[test_df["true_label"].eq(1)].sample(n=N_PER_CLASS, random_state=RANDOM_STATE)
sensitive_cases = test_df[test_df["true_label"].eq(0)].sample(n=N_PER_CLASS, random_state=RANDOM_STATE)

validation_30 = (
    pd.concat([resistant_cases, sensitive_cases], axis=0)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

validation_30.insert(0, "No", np.arange(1, len(validation_30) + 1))
validation_30["Keterangan"] = validation_30["true_label"].apply(label_text)

display(validation_30[["No", "Gene", "Mutation", "drug", "confidence", "Keterangan"]])


,No,Gene,Mutation,drug,confidence,Keterangan
0,1,embB,c.1710G>T,ethambutol,Not assoc w R - Interim,Sensitif
1,2,ndh,c.561A>C,delamanid,Not assoc w R - Interim,Sensitif
2,3,Rv1129c,c.960G>A,isoniazid,Not assoc w R - Interim,Sensitif
3,4,embA,c.784T>C,ethambutol,Not assoc w R - Interim,Sensitif
4,5,rpoB,p.Ser431_Met434del,rifampicin,Assoc w R - Interim,Resisten
5,6,rrl,n.2270G>T,linezolid,Assoc w R - Interim,Resisten
6,7,Rv0678,c.108G>T,bedaquiline,Not assoc w R - Interim,Sensitif
7,8,glpK,c.234C>T,moxifloxacin,Not assoc w R - Interim,Sensitif
8,9,pncA,p.Thr22fs,pyrazinamide,Assoc w R - Interim,Resisten
9,10,gid,p.Asn194fs,streptomycin,Assoc w R - Interim,Resisten


In [8]:
# Prediksi XGBoost-Tabular 3 fitur.
X_val_tabular = encoder.transform(validation_30[tabular_cols].astype(str))
tabular_pred = xgb_tabular.predict(X_val_tabular).astype(int)
tabular_prob = xgb_tabular.predict_proba(X_val_tabular)[:, 1]

# Prediksi XGBoost-ST+GNN Hybrid.
hybrid_model = joblib.load(HYBRID_MODEL_PATH)
X_val_hybrid = validation_30[feat_cols].to_numpy(dtype=float)
hybrid_pred = hybrid_model.predict(X_val_hybrid).astype(int)
hybrid_prob = hybrid_model.predict_proba(X_val_hybrid)[:, 1]

detail_df = validation_30[["No", "Gene", "Mutation", "drug", "confidence", "true_label", "Keterangan"]].copy()
detail_df = detail_df.rename(columns={"drug": "Drug", "true_label": "Label_Biner"})

detail_df["Prediksi XGBoost-Tabular"] = [label_text(v) for v in tabular_pred]
detail_df["Prob Resisten Tabular"] = np.round(tabular_prob, 4)
detail_df["Status Tabular"] = np.where(tabular_pred == validation_30["true_label"].to_numpy(), "Benar", "Salah")

detail_df["Prediksi XGBoost-ST+GNN (Hybrid)"] = [label_text(v) for v in hybrid_pred]
detail_df["Prob Resisten Hybrid"] = np.round(hybrid_prob, 4)
detail_df["Status Hybrid"] = np.where(hybrid_pred == validation_30["true_label"].to_numpy(), "Benar", "Salah")

display(detail_df)


,No,Gene,Mutation,Drug,confidence,Label_Biner,Keterangan,Prediksi XGBoost-Tabular,Prob Resisten Tabular,Status Tabular,Prediksi XGBoost-ST+GNN (Hybrid),Prob Resisten Hybrid,Status Hybrid
0,1,embB,c.1710G>T,ethambutol,Not assoc w R - Interim,0,Sensitif,Sensitif,0.0001,Benar,Sensitif,0.0022,Benar
1,2,ndh,c.561A>C,delamanid,Not assoc w R - Interim,0,Sensitif,Sensitif,0.0000,Benar,Sensitif,0.0018,Benar
2,3,Rv1129c,c.960G>A,isoniazid,Not assoc w R - Interim,0,Sensitif,Sensitif,0.0000,Benar,Sensitif,0.0051,Benar
3,4,embA,c.784T>C,ethambutol,Not assoc w R - Interim,0,Sensitif,Sensitif,0.0072,Benar,Sensitif,0.0003,Benar
4,5,rpoB,p.Ser431_Met434del,rifampicin,Assoc w R - Interim,1,Resisten,Sensitif,0.0818,Salah,Resisten,0.9990,Benar
5,6,rrl,n.2270G>T,linezolid,Assoc w R - Interim,1,Resisten,Sensitif,0.1766,Salah,Sensitif,0.4728,Salah
6,7,Rv0678,c.108G>T,bedaquiline,Not assoc w R - Interim,0,Sensitif,Sensitif,0.0077,Benar,Sensitif,0.0012,Benar
7,8,glpK,c.234C>T,moxifloxacin,Not assoc w R - Interim,0,Sensitif,Sensitif,0.0000,Benar,Sensitif,0.0007,Benar
8,9,pncA,p.Thr22fs,pyrazinamide,Assoc w R - Interim,1,Resisten,Resisten,0.9156,Benar,Resisten,0.9894,Benar
9,10,gid,p.Asn194fs,streptomycin,Assoc w R - Interim,1,Resisten,Sensitif,0.4097,Salah,Resisten,0.9979,Benar


In [9]:
def build_summary(model_name, status_col):
    correct = int((detail_df[status_col] == "Benar").sum())
    total = len(detail_df)
    wrong = total - correct
    return {
        "Model": model_name,
        "Kasus Testing": total,
        "Prediksi Benar": correct,
        "Prediksi Salah": wrong,
        "Akurasi Kasus": format_percent(correct / total),
    }


summary_df = pd.DataFrame(
    [
        build_summary("XGBoost-Tabular", "Status Tabular"),
        build_summary("XGBoost-ST+GNN (Hybrid)", "Status Hybrid"),
    ]
)

display(summary_df)


,Model,Kasus Testing,Prediksi Benar,Prediksi Salah,Akurasi Kasus
0,XGBoost-Tabular,30,26,4,"86,7%"
1,XGBoost-ST+GNN (Hybrid),30,29,1,"96,7%"


In [10]:
summary_path = OUTPUT_DIR / "testing30_summary_tabular_vs_hybrid.csv"
detail_path = OUTPUT_DIR / "testing30_detail_tabular_vs_hybrid.csv"

summary_df.to_csv(summary_path, index=False)
detail_df.to_csv(detail_path, index=False)

print("Saved:")
print(summary_path)
print(detail_path)


Saved:
C:\Semester 7\Project Tugas Akhir\Tugas Akhir\testing30_validation_outputs\testing30_summary_tabular_vs_hybrid.csv
C:\Semester 7\Project Tugas Akhir\Tugas Akhir\testing30_validation_outputs\testing30_detail_tabular_vs_hybrid.csv


## Catatan pelaporan

Tabel ringkasan di atas dapat dipakai seperti tabel revisi 8 kasus, tetapi diganti menjadi 30 kasus testing. Tabel detail menyertakan `Gene`, `Mutation`, `Drug`, label referensi `Resisten/Sensitif`, prediksi kedua model, probabilitas resisten, dan status benar/salah.
